# Streaming

<img src="./assets/LC_streaming.png" width="400">

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

## 环境准备

从仓库根目录加载并检查 `.env` 与依赖配置。

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# ルートの .env を読み込む
load_dotenv()

# ルート設定を確認する
doublecheck_env("example.env")

OLLAMA_BASE_URL=<not set>
OLLAMA_MODEL=<not set>


In [2]:
from langchain.agents import create_agent

In [3]:
agent = create_agent(
    model="ollama:qwen2.5:7b",
    system_prompt="You are a full-stack comedian",
)

## No Streaming (invoke)

In [4]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][1].content)

Why did the web developer go broke? Because he used up all his CSS!


## values
You have seen this streaming mode in our examples so far. 

In [5]:
# Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a Dad joke"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a Dad joke


================================== Ai Message ==================================

Why was the math book sad? Because it had too many problems.


## messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [6]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")

Sure thing! Here

's a light-hearted

, family-friendly

 poem for you:



In the Kitchen with

...

’re all happy and

 free,
A love

 for cooking that will

 last you till you're

 old and gray.

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer` writer allows you to easily stream `custom` data from sources you create.

In [7]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model="ollama:qwen2.5:7b",
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='6ef54aa2-e429-4874-b7e8-838fa19f253c')]})


('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='6ef54aa2-e429-4874-b7e8-838fa19f253c'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-03-07T09:26:59.549716Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1729456167, 'load_duration': 57951250, 'prompt_eval_count': 152, 'prompt_eval_duration': 672374334, 'eval_count': 20, 'eval_duration': 987349542, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--019cc79f-161b-7cf1-95ac-7ba76125f959-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': '9bf40835-4a83-4cc8-84c3-c96e43bc77bf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 20, 'total_tokens': 172})]})
('custom', 'Looking up data for city: SF')
('custom', 'Acquired data for city: SF')
('values', {'messages': [HumanMessage(content='What is th

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='6ef54aa2-e429-4874-b7e8-838fa19f253c'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b', 'created_at': '2026-03-07T09:26:59.549716Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1729456167, 'load_duration': 57951250, 'prompt_eval_count': 152, 'prompt_eval_duration': 672374334, 'eval_count': 20, 'eval_duration': 987349542, 'logprobs': None, 'model_name': 'qwen2.5:7b', 'model_provider': 'ollama'}, id='lc_run--019cc79f-161b-7cf1-95ac-7ba76125f959-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': '9bf40835-4a83-4cc8-84c3-c96e43bc77bf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 20, 'total_tokens': 172}), ToolMessage(content="It's always sunny in SF!", name='get_weather', id='fb92be44-f99b-4dd7-9024-b4f00e84755c', tool_call_id='9bf40835-4a83-4cc8

In [8]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up data for city: SF')
('custom', 'Acquired data for city: SF')


## Try different modes on your own!
Modify the stream mode and the select to produce different results.

In [9]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    if chunk[0] == "custom":
        print(chunk[1])

Looking up data for city: SF
Acquired data for city: SF
